In [1]:

from src.photo_management.metadata_preprocessing import *
from src.photo_management.photo_processing import *
from src.photo_management.db import *
import os


In [2]:
# Variables
photo_path = "Photos"
web_path = "temp/Web"
thumb_path = "temp/Thumb"
metadata_path = "temp/Metadata"

conn = get_connection()

In [3]:
# Get all photo path
photos = [
    photo_path + "/" + f
    for f in os.listdir(photo_path)
    if os.path.isfile(os.path.join(photo_path, f))
    and not f.startswith('.')
]

print(photos)

['Photos/精选-08.png', 'Photos/精选-03.png', 'Photos/精选-02.png', 'Photos/精选-01.png', 'Photos/精选-05.png', 'Photos/精选-04.png', 'Photos/精选-07.png']


In [4]:
path = "Photos/test/test.png"
xmp_str = get_xmp_str(path)
fields = extract_xmp_fields(xmp_str)
fields

{'title': None,
 'caption': None,
 'alt_text': None,
 'extended_description': None,
 'rating': '5',
 'creator_tool': 'Adobe Lightroom 8.4 (Macintosh)',
 'make': 'SONY',
 'model': 'ILCE-7RM4A',
 'lens': 'E 70-300mm F4.5-6.3 A047',
 'iso': '400',
 'exposure_time': '1/640',
 'fnumber': '63/10',
 'focal_length': '3000/10',
 'datetime_original': '2026-01-11T15:00:39-07:00',
 'lr_exposure2012': '+0.29',
 'state': 'Utah',
 'city': 'Moab',
 'country': 'United State of America'}

In [5]:
normalized_field = normalize_xmp_fields(fields)
normalized_field

{'title': None,
 'caption': None,
 'alt_text': None,
 'extended_description': None,
 'rating': '5',
 'creator_tool': 'Adobe Lightroom 8.4 (Macintosh)',
 'make': 'SONY',
 'model': 'ILCE-7RM4A',
 'lens': 'E 70-300mm F4.5-6.3 A047',
 'iso': 400,
 'exposure_time': '1/640',
 'fnumber': 6.3,
 'focal_length': 300.0,
 'datetime_original': '2026-01-11 15:00:39',
 'lr_exposure2012': '+0.29',
 'state': 'Utah',
 'city': 'Moab',
 'country': 'United State of America'}

In [6]:
insert_full_metadata(conn, path, normalized_field)

1

In [7]:
def fetch_all_photos(conn):
    cur = conn.cursor(dictionary=True)

    cur.execute("SELECT * FROM photos")
    rows = cur.fetchall()

    cur.close()

    return rows

In [9]:
p = fetch_all_photos(conn)

In [10]:
p

[{'id': 1,
  'original_path': 'Photos/test/test.png',
  'original_filename': 'test.png',
  'original_sha256': 'a9f72676ec4dbad3dd361d7b9b371f73b56c43d6d80180c19e49ca94b22571d5',
  'mime_type': None,
  'file_size': 36755606,
  'width': None,
  'height': None,
  'datetime_original': datetime.datetime(2026, 1, 11, 15, 0, 39),
  'created_at': datetime.datetime(2026, 9, 26, 0, 7, 22),
  'updated_at': datetime.datetime(2026, 9, 26, 0, 7, 22)},
 {'id': 2,
  'original_path': 'Photos/精选-08.png',
  'original_filename': '精选-08.png',
  'original_sha256': 'd0222c28403b7af2dd8e757d18c6dff3cab3501159b18a1b5c0083554e7b736a',
  'mime_type': None,
  'file_size': 37346986,
  'width': None,
  'height': None,
  'datetime_original': datetime.datetime(2024, 10, 26, 19, 1, 26),
  'created_at': datetime.datetime(2026, 9, 26, 0, 8, 35),
  'updated_at': datetime.datetime(2026, 9, 26, 0, 8, 35)},
 {'id': 3,
  'original_path': 'Photos/精选-03.png',
  'original_filename': '精选-03.png',
  'original_sha256': '4d0403db2d

In [11]:
photo_path = "Photos"
web_path = "temp/Web"
thumb_path = "temp/Thumb"
metadata_path = "temp/Metadata"

photos = [
    os.path.join(photo_path, f)
    for f in os.listdir(photo_path)
    if os.path.isfile(os.path.join(photo_path, f))
    and not f.startswith(".")
]

for photo in photos:
    sha256 = sha256_file(photo)

    # Check database for duplicate
    existing_id = get_photo_id_by_sha256(conn, sha256)

    if existing_id is not None:
        print(f"Skip ALL processing (duplicate): {photo}")
        continue

    # Extract metadata
    xmp_str = get_xmp_str(photo)
    export_metadata_to_txt(photo, metadata_path)

    field = extract_xmp_fields(xmp_str)
    normalized_field = normalize_xmp_fields(field)

    # Generate variants
    export_web_jpg(photo, web_path)
    export_thumb_jpg(photo, thumb_path)

    # Insert into database
    insert_full_metadata(
        conn,
        photo,
        normalized_field
    )

print(fetch_all_photos(conn))

Skip ALL processing (duplicate): Photos/精选-08.png
Skip ALL processing (duplicate): Photos/精选-03.png
Skip ALL processing (duplicate): Photos/精选-02.png
Skip ALL processing (duplicate): Photos/精选-01.png
Skip ALL processing (duplicate): Photos/精选-05.png
Skip ALL processing (duplicate): Photos/精选-04.png
Skip ALL processing (duplicate): Photos/精选-07.png
[{'id': 1, 'original_path': 'Photos/test/test.png', 'original_filename': 'test.png', 'original_sha256': 'a9f72676ec4dbad3dd361d7b9b371f73b56c43d6d80180c19e49ca94b22571d5', 'mime_type': None, 'file_size': 36755606, 'width': None, 'height': None, 'datetime_original': datetime.datetime(2026, 1, 11, 15, 0, 39), 'created_at': datetime.datetime(2026, 9, 26, 0, 7, 22), 'updated_at': datetime.datetime(2026, 9, 26, 0, 7, 22)}, {'id': 2, 'original_path': 'Photos/精选-08.png', 'original_filename': '精选-08.png', 'original_sha256': 'd0222c28403b7af2dd8e757d18c6dff3cab3501159b18a1b5c0083554e7b736a', 'mime_type': None, 'file_size': 37346986, 'width': None, 'he